# SVC — Targeted Weight & Augmentation

B/D/C 대상 가중치 및 증강량 비교. TW2 Full 별도.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("SVC_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("SVC", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
sbase=base
print("Local output:",root)

In [ ]:
t=time.perf_counter()
Xs,ys,meta=fb.train_features('SVC_Combined',off,wr)
print(Xs.shape,meta,time.perf_counter()-t)
np.save(root/'runtime/svc_X.npy',Xs)
np.save(root/'runtime/svc_y.npy',ys)
np.save(root/'runtime/dev_X.npy',Xv)
np.save(root/'runtime/dev_y.npy',y)
np.save(root/'runtime/official_X.npy',Xo)
np.save(root/'runtime/official_y.npy',yo)

In [ ]:
# Predeclare pilot for resource-limited SVC screen (explicit not full DOE). 
# Save exact training selections and augmentation candidate (no validation used in generation).
rng=np.random.default_rng(42)
# sample balanced 500 per class official, 4 per class/cohort per origin/aug, 2 per class/cohort for targeted
pilot=[]
for c0 in range(16):
 pilot.extend(rng.choice(np.flatnonzero(ys[:76800]==c0),500,replace=False))
for offset in [76800,79360]:
 for w0 in range(4):
  for c0 in range(16):
   pool=np.flatnonzero(yh[w0*640:(w0+1)*640]==c0)+w0*640
   pilot.extend(offset+rng.choice(pool,4,replace=False))
# exact target source indices used by baseline
rngt=np.random.default_rng(fb.TARGET_SEED)
target_groups=[]
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)==w)&(yh==c0))
  target_groups.append(rngt.permutation(pool)[:20])
ti=np.sort(np.concatenate(target_groups))
assert np.array_equal(ys[81920:],yh[ti])
# four-cohort target sampled 2 per class/cohort
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)[ti]==w)&(yh[ti]==c0))
  pilot.extend(81920+rng.choice(pool,2,replace=False))
pilot=np.sort(np.array(pilot))
assert len(pilot)==8528
# predeclared pilot official eval balanced 40/class =640 (not full)
oid=np.sort(np.concatenate([rng.choice(np.flatnonzero(yo==c0),40,replace=False) for c0 in range(16)]))
np.savez(root/'02_SVC/pilot_indices.npz',training_rows=pilot,official_validation_rows=oid)
proto['SVC']['pilot_resource_adaptation']={'rows':len(pilot),'official_train':8000,'handmade_original':256,'handmade_augmented':256,'handmade_targeted':16,'official_validation':640,'handmade_validation':1280,'purpose':'screening only; full 9x DOE not claimed complete; full fit required for adoption'}
(root/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(proto,ensure_ascii=False,indent=2))
print(len(pilot),len(oid),np.bincount(ys[pilot]))
# set up data 
Xp=Xs[pilot];yp=ys[pilot];hp=(pilot>=76800)
gscale=1/(Xp.shape[1]*Xp.var())
print('pilot gamma',gscale)

In [ ]:
from sklearn.svm import SVC
spdir=root/'02_SVC'
sprows=[]
sppreds={}
def svc_pilot(name,exp,C=1.,gm=1.,hand_w=1.,target_w=1.,xx=Xp,yy=yp,handmask=hp):
    t=time.perf_counter()
    gamma=float(gm/(xx.shape[1]*xx.var()))
    weights=np.ones(len(yy),np.float64); weights[handmask]*=hand_w
    weights[handmask & np.isin(yy,[labels.index(c) for c in 'BDC'])]*=target_w
    m=SVC(C=C,gamma=gamma,cache_size=200)
    m.fit(xx,yy,sample_weight=weights)
    fit=time.perf_counter()-t
    t=time.perf_counter(); pv=m.predict(Xv); vo=m.predict(Xo[oid]); infer=time.perf_counter()-t
    res={'condition':name,'experiment':exp,'scope':'pilot','train_rows':len(yy),'C':C,'gamma_multiplier':gm,'gamma':gamma,'hand_weight':hand_w,'target_weight':target_w,'fit_s':fit,'batch_eval_s':infer,'official_pilot_n':len(oid),'official_pilot_correct':int((vo==yo[oid]).sum()),**stats(pv,sbase)}
    sprows.append(res);sppreds[name]=pv
    pd.DataFrame(sprows).to_csv(spdir/'pilot_all_conditions.csv',index=False)
    pd.DataFrame({'true':y,**sppreds}).to_csv(spdir/'pilot_predictions.csv',index=False)
    print(name,res['correct'],round(res['macro_f1'],5),res['official_pilot_correct'],round(fit,2),flush=True)
    return res

In [ ]:
svc_pilot("C1_G1","001")

In [ ]:
for w in [1.5,2]:
 svc_pilot(f'TW{w}','003',target_w=w)

In [ ]:
# Additional augmentation controlled comparison: 80 selected rows instead of 160, or +240 new B/D/C variants.
# Baseline indices, reduction pilot exactly one retained from two per B/D cohort, augmentation generation from existing training only.
rng2=np.random.default_rng(20261128)
target_base_pilot=pilot[pilot>=81920]
keep_red=[]
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  cand=target_base_pilot[(np.repeat(fb.W,640)[ti[target_base_pilot-81920]]==w)&(ys[target_base_pilot]==c0)]
  assert len(cand)==2
  keep_red.append(cand[0])
rp=np.r_[pilot[pilot<81920],np.sort(keep_red)]
svc_pilot("TA_reduced","003",xx=Xs[rp],yy=ys[rp],handmask=rp>=76800)
# prepare full new target augment 20 per cohort per B,D,C, distinct random variant seed; preserve existing base
add_groups=[]
for w in fb.W:
 for c0 in [labels.index(c) for c in 'BDC']:
  pool=np.flatnonzero((np.repeat(fb.W,640)==w)&(yh==c0))
  add_groups.append(rng2.permutation(pool)[:20])
add_idx=np.sort(np.concatenate(add_groups))
add_feat=fb.hand_aug(rawh,add_idx,20261128+1000)
add_y=yh[add_idx]
assert add_feat.shape==(240,1108)
np.savez(root/'runtime/svc_target_extra.npz',X=add_feat,y=add_y,source_indices=add_idx)
addpilot=[]
for w in fb.W:
 for c0 in [labels.index(c) for c in 'BDC']:
  pool=np.flatnonzero((np.repeat(fb.W,640)[add_idx]==w)&(add_y==c0))
  addpilot.extend(rng2.choice(pool,2,replace=False))
addpilot=np.sort(addpilot)
svc_pilot("TA_increased","003",xx=np.vstack([Xp,add_feat[addpilot]]),yy=np.r_[yp,add_y[addpilot]],handmask=np.r_[hp,np.ones(len(addpilot),bool)])
# Record actual same-scope pilot stats vs pilot baseline
sp=pd.DataFrame(sprows); basep=sp[sp.condition=='C1_G1'].iloc[0]
sp['delta_correct_vs_pilot']=sp.correct-basep.correct
sp['delta_f1_vs_pilot']=sp.macro_f1-basep.macro_f1
sp['pilot_guardrail']= (sp.correct>basep.correct)&(sp.macro_precision>=basep.macro_precision)&(sp.macro_f1>basep.macro_f1)&(sp.official_pilot_correct>=basep.official_pilot_correct-1)
sp.to_csv(spdir/'pilot_all_conditions.csv',index=False)
print(sp[['condition','correct','macro_precision','macro_f1','official_pilot_correct','pilot_guardrail']].to_string(index=False))

### Full 후보 실행 (선택)
Pilot과 Full은 구분한다. 원본 Full 스크립트 3개는 `07_Final_Model/Code/`에 있다. 아래 셀은 해당 스크립트를 로컬 결과 폴더로 복사만 한다. 실제 실행 명령은 최종 모델 README 참조. 기존 Full 실험은 C2_G2/SW4/TW2/SW1.5만 완료.

In [ ]:
for name in ["run_svc_full.py","run_svc_remaining.py","run_svc_sw15_followup.py"]:
    shutil.copy2(FT/"01_SVC"/"07_Final_Model"/"Code"/name,root/name)
print("Full scripts copied; runtime arrays prepared in",root)